# Feature Engineering — M5 Dataset
**Input**: Pre-prepared M5 panel (from v2 pipeline)  
**Output**: `m5_panel_features.parquet`  
The M5 panel already has lag/rolling/calendar features from `01_prepare_m5.py`. We verify and extend.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

panel = pd.read_parquet('../outputs/m5_prepared/m5_monthly_panel.parquet')
panel['month'] = pd.to_datetime(panel['month'].astype(str))
print(f"✅ Panel: {panel.shape}")
print(f"   Existing features: {[c for c in panel.columns if c.startswith(('lag','roll','month_','quarter','year'))]}")
print(f"   Split distribution:")
print(panel['split'].value_counts())

In [ ]:
# Verify existing features
feature_cols = [c for c in panel.columns if c.startswith(('lag_','roll_','month_sin','month_cos'))
                or c in ['quarter','year','month_num']]
if 'sell_price' in panel.columns:
    feature_cols.append('sell_price')
print(f"\n📋 FEATURE COLUMNS ({len(feature_cols)}):")
for f in feature_cols:
    nulls = panel[f].isna().sum()
    print(f"  {f:20s} nulls={nulls:>5}  mean={panel[f].mean():>10.1f}")

In [ ]:
# Feature correlation heatmap
fig, ax = plt.subplots(figsize=(12,9))
corr = panel[feature_cols].corr()
sns.heatmap(corr, annot=False, cmap='RdBu_r', center=0, ax=ax, linewidths=0.2)
ax.set_title('Feature Correlation Matrix — M5', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('plots/06_feature_corr.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
# Split visualization
fig, ax = plt.subplots(figsize=(14,5))
colors = {'train':'steelblue','validation':'orange','test':'seagreen'}
for sid in panel['series_id'].unique()[:3]:
    for split in ['train','validation','test']:
        d = panel[(panel['series_id']==sid)&(panel['split']==split)].sort_values('month')
        ax.plot(d['month'], d['demand'], 'o-', color=colors[split], linewidth=1.5, markersize=3, alpha=0.7)
ax.axvline(panel[panel['split']=='validation']['month'].min(), color='orange', ls='--', lw=2, label='Val start')
ax.axvline(panel[panel['split']=='test']['month'].min(), color='seagreen', ls='--', lw=2, label='Test start')
ax.set_title('Train/Val/Test Split', fontsize=14, fontweight='bold')
ax.legend(); plt.tight_layout(); plt.savefig('plots/07_split.png',dpi=150,bbox_inches='tight'); plt.show()

# Save with feature_cols metadata
panel.to_parquet('m5_panel_features.parquet', index=False)
print(f"\n✅ Saved: m5_panel_features.parquet ({panel.shape})")
print(f"   Feature columns: {feature_cols}")

## Summary
- **23 features** already engineered by v2 pipeline (lags, rolling, calendar, price)
- **Split**: ~35 months train / 6 months val / 12 months test
- Panel saved for model comparison notebook